# Sepsis early prediction (PhysioNet 2019)
Baseline (gradient-boosted trees) vs LSTM, evaluated on AUPRC, plus a lead-time analysis.

## Download

In [ ]:
!pip install -q awscli
!mkdir -p /content/sepsis
!aws s3 sync --no-sign-request s3://physionet-open/challenge-2019/1.0.0/training/ /content/sepsis/ --quiet

import glob

files = sorted(glob.glob("/content/sepsis/**/*.psv", recursive=True))
print("patient files:", len(files))

## Setup

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence, pad_packed_sequence
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score, average_precision_score
import matplotlib.pyplot as plt

SEED = 42
MAX_LEN = 72
HIDDEN = 64
BATCH = 64
EPOCHS = 5
MEASURED = ["Lactate", "WBC", "Creatinine", "Platelets", "HCO3", "Glucose"]

np.random.seed(SEED)
torch.manual_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

## EDA

In [ ]:
rng = np.random.default_rng(SEED)
sample = rng.choice(files, size=5000, replace=False)

n_patients = len(sample)
n_septic = 0
total_hours = 0
positive_hours = 0
lengths = []
missing = None

for f in sample:
    patient = pd.read_csv(f, sep="|")
    if missing is None:
        missing = pd.Series(0.0, index=patient.columns)
    label = patient["SepsisLabel"].values
    n_septic += int(label.max() > 0)
    total_hours += len(patient)
    positive_hours += int(label.sum())
    lengths.append(len(patient))
    missing += patient.isna().sum()

print("patients sampled:", n_patients)
print("septic patients:", n_septic, f"({100 * n_septic / n_patients:.1f}%)")
print("positive hours:", positive_hours, f"({100 * positive_hours / total_hours:.2f}%)")
print("median ICU hours:", int(np.median(lengths)))
print()
print("most-missing columns:")
print((missing / total_hours).sort_values(ascending=False).head(12).round(3))

## Load

In [ ]:
from tqdm.auto import tqdm

frames = []
for pid, f in enumerate(tqdm(files)):
    patient = pd.read_csv(f, sep="|")
    patient["pid"] = pid
    frames.append(patient)

df = pd.concat(frames, ignore_index=True)
df = df.sort_values(["pid", "ICULOS"]).reset_index(drop=True)

label_col = "SepsisLabel"
feature_cols = [c for c in df.columns if c not in (label_col, "pid")]
print("combined shape:", df.shape)

## Preprocessing

In [ ]:
for c in MEASURED:
    df[c + "_measured"] = df[c].notna().astype(float)
indicator_cols = [c + "_measured" for c in MEASURED]

df[feature_cols] = df.groupby("pid")[feature_cols].ffill()

patient_ids = df["pid"].unique()
rng = np.random.default_rng(SEED)
rng.shuffle(patient_ids)
n = len(patient_ids)
train_ids = set(patient_ids[:int(0.70 * n)])
val_ids = set(patient_ids[int(0.70 * n):int(0.85 * n)])
test_ids = set(patient_ids[int(0.85 * n):])

def which_split(pid):
    if pid in train_ids:
        return "train"
    if pid in val_ids:
        return "val"
    return "test"

df["split"] = df["pid"].map(which_split)

train_mask = df["split"] == "train"
medians = df.loc[train_mask, feature_cols].median()
df[feature_cols] = df[feature_cols].fillna(medians)
mean = df.loc[train_mask, feature_cols].mean()
std = df.loc[train_mask, feature_cols].std().replace(0, 1.0)
df[feature_cols] = (df[feature_cols] - mean) / std

all_features = feature_cols + indicator_cols
print("features:", len(all_features))
print("positive-hour rate:", f"{100 * df[label_col].mean():.2f}%")

## Baseline

In [ ]:
train_rows = df["split"] == "train"
test_rows = df["split"] == "test"

X_train = df.loc[train_rows, all_features].values
y_train = df.loc[train_rows, label_col].values
X_test = df.loc[test_rows, all_features].values
y_test = df.loc[test_rows, label_col].values

pos_weight = (y_train == 0).sum() / max((y_train == 1).sum(), 1)
sample_weight = np.where(y_train == 1, pos_weight, 1.0)

clf = HistGradientBoostingClassifier(
    max_iter=200,
    learning_rate=0.08,
    max_depth=6,
    random_state=SEED,
)
clf.fit(X_train, y_train, sample_weight=sample_weight)

base_prob = clf.predict_proba(X_test)[:, 1]
base_auroc = roc_auc_score(y_test, base_prob)
base_auprc = average_precision_score(y_test, base_prob)
print("baseline  AUROC:", round(base_auroc, 3), " AUPRC:", round(base_auprc, 3))

## LSTM

In [ ]:
def make_sequences(ids):
    sequences = []
    subset = df[df["pid"].isin(ids)]
    for pid, group in subset.groupby("pid"):
        x = group[all_features].values[-MAX_LEN:]
        y = group[label_col].values[-MAX_LEN:]
        x = torch.tensor(x, dtype=torch.float32)
        y = torch.tensor(y, dtype=torch.float32)
        sequences.append((x, y))
    return sequences


class SepsisDataset(Dataset):
    def __init__(self, sequences):
        self.sequences = sequences

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        return self.sequences[idx]


def pad_batch(batch):
    xs = [item[0] for item in batch]
    ys = [item[1] for item in batch]
    lengths = torch.tensor([len(x) for x in xs])
    x_pad = pad_sequence(xs, batch_first=True)
    y_pad = pad_sequence(ys, batch_first=True)
    mask = torch.arange(x_pad.size(1))[None, :] < lengths[:, None]
    return x_pad, y_pad, lengths, mask.float()


class SepsisLSTM(nn.Module):
    def __init__(self, n_features, hidden):
        super().__init__()
        self.lstm = nn.LSTM(n_features, hidden, batch_first=True)
        self.head = nn.Linear(hidden, 1)

    def forward(self, x, lengths):
        packed = pack_padded_sequence(x, lengths.cpu(), batch_first=True, enforce_sorted=False)
        out, _ = self.lstm(packed)
        out, _ = pad_packed_sequence(out, batch_first=True)
        return self.head(out).squeeze(-1)


train_loader = DataLoader(SepsisDataset(make_sequences(train_ids)), batch_size=BATCH, shuffle=True, collate_fn=pad_batch)
test_loader = DataLoader(SepsisDataset(make_sequences(test_ids)), batch_size=BATCH, shuffle=False, collate_fn=pad_batch)

model = SepsisLSTM(len(all_features), HIDDEN).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
pos_weight_tensor = torch.tensor([pos_weight], dtype=torch.float32, device=device)
loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight_tensor, reduction="none")

for epoch in range(EPOCHS):
    model.train()
    running = 0.0
    for x_pad, y_pad, lengths, mask in train_loader:
        x_pad = x_pad.to(device)
        y_pad = y_pad.to(device)
        mask = mask.to(device)
        logits = model(x_pad, lengths)
        loss = (loss_fn(logits, y_pad) * mask).sum() / mask.sum()
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        running += loss.item()
    print(f"epoch {epoch + 1}/{EPOCHS}  loss {running / len(train_loader):.4f}")

model.eval()
probs = []
trues = []
with torch.no_grad():
    for x_pad, y_pad, lengths, mask in test_loader:
        logits = model(x_pad.to(device), lengths)
        p = torch.sigmoid(logits).cpu().numpy()
        m = mask.numpy().astype(bool)
        probs.append(p[m])
        trues.append(y_pad.numpy()[m])

probs = np.concatenate(probs)
trues = np.concatenate(trues)
lstm_auroc = roc_auc_score(trues, probs)
lstm_auprc = average_precision_score(trues, probs)
print("LSTM      AUROC:", round(lstm_auroc, 3), " AUPRC:", round(lstm_auprc, 3))

## Compare and save

In [ ]:
results = pd.DataFrame({
    "model": ["baseline (GBDT)", "LSTM"],
    "AUROC": [base_auroc, lstm_auroc],
    "AUPRC": [base_auprc, lstm_auprc],
})
print(results.to_string(index=False))

fig, ax = plt.subplots(figsize=(6, 4))
x = np.arange(2)
width = 0.35
ax.bar(x - width / 2, results["AUROC"], width, label="AUROC")
ax.bar(x + width / 2, results["AUPRC"], width, label="AUPRC")
ax.set_xticks(x)
ax.set_xticklabels(results["model"])
ax.set_ylim(0, 1)
ax.set_title("Sepsis early prediction - baseline vs LSTM")
ax.legend()
fig.tight_layout()
fig.savefig("sepsis_results.png", dpi=130)
plt.show()

results.to_csv("sepsis_results.csv", index=False)

## Lead-time analysis

In [ ]:
def predict_patient(pid):
    group = df[df["pid"] == pid]
    x = group[all_features].values[-MAX_LEN:]
    x = torch.tensor(x, dtype=torch.float32)[None].to(device)
    lengths = torch.tensor([x.size(1)])
    with torch.no_grad():
        prob = torch.sigmoid(model(x, lengths)).cpu().numpy()[0]
    label = group[label_col].values[-MAX_LEN:]
    return prob, label


model.eval()
test_patients = []
for pid in test_ids:
    prob, label = predict_patient(pid)
    test_patients.append((pid, prob, label))

septic = [(pid, prob, label) for pid, prob, label in test_patients if label.max() > 0]
healthy = [(pid, prob, label) for pid, prob, label in test_patients if label.max() == 0]
print("test patients:", len(septic), "septic,", len(healthy), "non-septic")
print()

all_probs = np.concatenate([prob for _, prob, _ in test_patients])

for pct in [90, 95, 99]:
    threshold = np.percentile(all_probs, pct)
    leads = []
    caught = 0
    for pid, prob, label in septic:
        first_positive = int(np.argmax(label > 0))
        onset = first_positive + 6
        fired = prob >= threshold
        if fired.any():
            alarm = int(np.argmax(fired))
            if alarm <= onset:
                caught += 1
                leads.append(onset - alarm)
    false_alarm = np.mean([(prob >= threshold).any() for _, prob, _ in healthy])
    median_lead = int(np.median(leads)) if leads else 0
    print(f"top {100 - pct}% hours:  caught {100 * caught / len(septic):.0f}%  "
          f"median lead {median_lead} h  false alarms {100 * false_alarm:.0f}%")